# Enhanced Summary Knowledge Tuning - Data Generation

## Overview

This notebook demonstrates how to generate high-quality knowledge tuning datasets using the SDG Hub framework. It creates multiple types of document augmentations and corresponding question-answer pairs that can be used to train or fine-tune language models for enhanced summarization and knowledge extraction capabilities.

## What This Notebook Does

This notebook will:

2. **Generate Four Types of Knowledge Tuning Datasets**:
   - **Extractive Summaries**: Concise summaries that extract key information directly from source documents
   - **Detailed Summaries**: Comprehensive summaries that provide thorough coverage of document content
   - **Key Facts**: Structured fact extraction with corresponding Q&A pairs
   - **Document-Based Q&A**: Question-answer pairs generated directly from document content


4. **Output Structured Training Data**:
   - For each augmentation we save JSONL dataset.
   - You can follow [knowledge_mixing](knowledge_mixing.ipynb) to convert it into training dataset

## Prerequisites

- SDG Hub installed and configured
- Environment variables set up (see [.env.example](.env.example)). Specifically set the model provider, seed data and output path.
- Document pre-processing completed (run [document_pre_processing.ipynb](document_pre_processing.ipynb) first)

```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
copy the .env.example to .env and set the model endpoint and generation/mixing parameters
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

## Next Steps

After running this notebook, use [knowledge_mixing](knowledge_mixing.ipynb) to combine and curate the generated datasets for final model training.


In [1]:
!unset GROQ_API_KEY

In [2]:
print('hello')

hello


In [3]:
# Third Party
from datasets import load_dataset
from dotenv import load_dotenv

# First Party
from sdg_hub import Flow, FlowRegistry
from sdg_hub.core.utils.translation import translate_flow
import os
from pathlib import Path

# Load environment variables from .env file
load_dotenv(override=True)

True

In [4]:
# Load seed data. If one is not provided, create it from the quality benchmark dataset.
seed_data_path = 'data/seed_data.jsonl'

print(f"Loading existing seed data from {seed_data_path}")
quality_corpus = load_dataset("json", data_files=seed_data_path, split="train")
quality_corpus = quality_corpus.to_pandas()

Loading existing seed data from data/seed_data.jsonl


### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [5]:
quality_corpus = quality_corpus.sample(10)

In [6]:
!curl http://localhost:8102/v1/models

curl: (7) Failed to connect to localhost port 8102: Connection refused


In [ ]:
# Setup model configuration in flow object
def set_model_config(flow_object):
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    timeout_seconds = 600.0
    print(f"Using model provider: {model_provider}")
    # Set model provider
    if model_provider == "hosted_vllm":
        vllm_model = os.getenv(
            "VLLM_MODEL", "hosted_vllm/glm-4.7-fp8"
        )
        vllm_api_base = os.getenv("VLLM_API_BASE", "http://10.241.128.25:8100/v1")
        vllm_api_key = os.getenv("VLLM_API_KEY", "EMPTY")
        enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in (
            "1",
            "true",
            "yes",
        )
        print(f"Using reasoning: {enable_reasoning}")
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
            enable_reasoning=enable_reasoning,
            timeout=timeout_seconds,
        )
    elif model_provider == "openai":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
            timeout=timeout_seconds,
        )
    elif model_provider == "openrouter":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
            api_base="https://openrouter.ai/api/v1",
            timeout=timeout_seconds,
        )
    elif model_provider == "ollama":
        ollama_model = os.getenv("OLLAMA_MODEL", "ollama/gemma2")
        ollama_api_base = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")
        flow_object.set_model_config(
            model=ollama_model,
            api_base=ollama_api_base,
            timeout=timeout_seconds,
        )
    elif model_provider == "maas":
        maas_model = os.getenv("MAAS_MODEL")
        maas_api_base = os.getenv("MAAS_API_BASE")
        maas_api_key = os.getenv("MAAS_API_KEY")
        flow_object.set_model_config(
            model=maas_model,
            api_base=maas_api_base,
            api_key=maas_api_key,
            timeout=timeout_seconds,
        )
    elif model_provider == "groq":
        groq_api_key = os.getenv("GROQ_API_KEY")
        groq_api_base = os.getenv("GROQ_API_BASE", "https://api.groq.com/openai/v1")
        groq_model = os.getenv("GROQ_MODEL", "groq/qwen/qwen3-32b")
        print(groq_api_key)
        flow_object.set_model_config(
            model=groq_model,
            api_key=groq_api_key,
            api_base=groq_api_base,
            timeout=timeout_seconds,
        )
    
    return flow_object

#### Discover the available generation flows

In [22]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                  ┃ Name                 ┃ Author               ┃ Tags                 ┃ Description          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397    │ Advanced Japanese    │ SDG Hub Contributors │ question-generation, │ A comprehensive flow │
│                     │ Document Grounded    │                      │ knowledge-extractio… │ that generates       │
│                     │ Question-Answer      │                      │ qa-pairs,            │ high-quality         │
│                     │ Generation Flow for  │                      │ document-processing, │ question-answer      │
│                     │ Knowledge Tuning     │                      │ educational,         │ pairs from Japanese  │
│                     │                      │                      │ multilingual,        │ input documents      │
│                     │                      │                      │ japanese             │ using multiple LLM   │
│                     │                      │                      │                      │ blocks for question  │
│                     │                      │                      │                      │ generation, answer   │
│                     │                      │                      │                      │ synthesis, and       │
│                     │                      │                      │                      │ quality evaluation.  │
│ epic-jade-656       │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document. Each │
│                     │ Flow                 │                      │ knowledge-extractiv… │ document is first    │
│                     │                      │                      │ qa-pairs,            │ converted into list  │
│                     │                      │                      │ extractive-summaries │ of knowledge         │
│                     │                      │                      │                      │ segments for         │
│                     │                      │                      │                      │ creating extractive  │
│                     │                      │                      │                      │ summary and then     │
│                     │                      │                      │                      │ annotated with       │
│                     │                      │                      │                      │ context,             │
│                     │                      │                      │                      │ relationship and     │
│                     │                      │                      │                      │ relevance. This is   │
│                     │                      │                      │                      │ then converted into  │
│                     │                      │                      │                      │ Question-Answer      │
│                     │                      │                      │                      │ pairs.               │
│ epic-jade-656-es    │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document in    │
│                     │ Flow (Spanish)       │                      │ knowledge-extractiv… │ Spanish. Each        │
│                     │                      │          

Available flows: [{'id': 'clean-shadow-397', 'name': 'Advanced Japanese Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'stellar-peak-605-es', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'heavy-heart-77-es', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'mild-thunder-748-es', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'epic-jade-656-es', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'stellar-peak-605', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'green-clay-812', 'name': 'Structured Text 

#### Multilingual Support (Optional)

If `SDG_LANG` and `SDG_LANG_CODE` are set in your `.env` file, the notebook will
use translated flow variants. If translated flows are not yet available in the
FlowRegistry, they will be created automatically using `translate_flow()`.

Delete `SDG_LANG` from your `.env` to use the default English flows.

In [23]:
# Read multilingual settings from .env
sdg_lang = os.getenv("SDG_LANG", "").strip()
sdg_lang_code = os.getenv("SDG_LANG_CODE", "").strip()

if sdg_lang and not sdg_lang_code:
    raise ValueError("SDG_LANG is set but SDG_LANG_CODE is missing. Both are required.")
if sdg_lang_code and not sdg_lang:
    raise ValueError("SDG_LANG_CODE is set but SDG_LANG is missing. Both are required.")

# If a custom translated flows directory exists, register it for discovery
translated_flows_dir = os.getenv("TRANSLATED_FLOWS_DIR", "").strip()
if translated_flows_dir and os.path.isdir(translated_flows_dir):
    FlowRegistry.register_search_path(translated_flows_dir)
    FlowRegistry._discover_flows(force_refresh=True)


def ensure_translated_flow(base_flow_name: str) -> str:
    """Return the flow name to use, translating on-demand if needed.

    When SDG_LANG is unset the base (English) name is returned unchanged.
    Otherwise checks FlowRegistry for a pre-existing translated variant
    and falls back to running translate_flow() to create one.
    """
    if not sdg_lang:
        return base_flow_name

    translated_name = f"{base_flow_name} ({sdg_lang})"
    if FlowRegistry.get_flow_path(translated_name) is not None:
        print(f"  ✓ Found: {translated_name}")
        return translated_name

    # Translate on-demand
    print(f"  ⟳ Translating: {base_flow_name} → {sdg_lang}...")
    # Compute per-flow output subdir to avoid flow.yaml collisions
    source_path = FlowRegistry.get_flow_path(base_flow_name)
    flow_subdir = Path(source_path).parent.name
    output_dir = None
    if translated_flows_dir:
        output_dir = str(Path(translated_flows_dir) / f"{flow_subdir}_{sdg_lang_code}")

    translate_flow(
        flow=base_flow_name,
        lang=sdg_lang,
        lang_code=sdg_lang_code,
        translator_model=os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o"),
        verifier_model=os.getenv(
            "VERIFIER_MODEL", os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o")
        ),
        output_dir=output_dir,
        translator_api_key=os.getenv("TRANSLATOR_API_KEY"),
        translator_api_base=os.getenv("TRANSLATOR_API_BASE"),
        verifier_api_key=os.getenv("VERIFIER_API_KEY"),
        verifier_api_base=os.getenv("VERIFIER_API_BASE"),
        register=True,
    )
    return translated_name


# Resolve all flow names (translating if needed)
BASE_FLOW_NAMES = [
    "Extractive Summary Knowledge Tuning Dataset Generation Flow",
    "Detailed Summary Knowledge Tuning Dataset Generation Flow",
    "Key Facts Knowledge Tuning Dataset Generation Flow",
    "Document Based Knowledge Tuning Dataset Generation Flow",
]

if sdg_lang:
    print(f"Multilingual mode: {sdg_lang} ({sdg_lang_code})\n")
else:
    print("Language: English (default)\n")

flow_names = {}
for name in BASE_FLOW_NAMES:
    flow_names[name] = ensure_translated_flow(name)

print(f"\nFlows to use: {list(flow_names.values())}")

Language: English (default)


Flows to use: ['Extractive Summary Knowledge Tuning Dataset Generation Flow', 'Detailed Summary Knowledge Tuning Dataset Generation Flow', 'Key Facts Knowledge Tuning Dataset Generation Flow', 'Document Based Knowledge Tuning Dataset Generation Flow']


In [33]:
# Get runtime parameters
enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in (
    "1",
    "true",
    "yes",
)
number_of_summaries = int(os.getenv("NUMBER_OF_SUMMARIES", "50"))
max_concurrency = int(os.getenv("MAX_CONCURRENCY", "8"))
save_data_path = os.getenv("OUTPUT_DATA_FOLDER", "")
generate_cpt = os.getenv("GENERATE_CPT", "false").lower() in (
    "1",
    "true",
    "yes",
)

In [25]:
flow_name = flow_names["Extractive Summary Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow_path

'/workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_summary/flow.yaml'

In [12]:
!cat '/workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_summary/flow.yaml'

metadata:
  name: Extractive Summary Knowledge Tuning Dataset Generation Flow
  description: Generate extractive summary from the input document. Each document is first converted into list of knowledge segments for creating extractive summary and then annotated with context, relationship and relevance. This is then converted
    into Question-Answer pairs.
  version: 2.0.0
  author: SDG Hub Contributors
  recommended_models:
    default: openai/gpt-oss-120b
    compatible:
    - meta-llama/Llama-3.3-70B-Instruct
    - microsoft/phi-4
    - mistralai/Mixtral-8x7B-Instruct-v0.1
    experimental: []
  tags:
  - knowledge-tuning
  - document-internalization
  - question-generation
  - knowledge-extractive-summary
  - qa-pairs
  - extractive-summaries
  license: Apache-2.0
  dataset_requirements:
    required_columns:
    - document
    - document_outline
    - domain
    - icl_document
    - icl_query_1
    - icl_query_2
    - icl_query_3
    description: 'Input dataset should contain docu

In [13]:
import nest_asyncio; nest_asyncio.apply()

In [26]:
import os
os.environ['MODEL_PROVIDER'] = 'groq'
# os.environ['GROQ_API_BASE'] = 'https://api.groq.com/openai/v1'

In [27]:
bool(os.environ['GROQ_API_KEY'])

True

In [34]:
# Generate data for extractive summary
flow_name = flow_names["Extractive Summary Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)


[18:10:38] INFO     Loading flow from:                                                          ]8;id=101426;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=629973;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py#60\60]8;;\
                    /workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-pac                    
                    kages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive                    
                    _summary/flow.yaml                                                                             

[18:10:38] INFO     Auto-detected 4 LLM blocks for configuration: ['answer_generation',         ]8;id=139662;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=177891;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'eval_faithful_llm_chat', 'gen_extractive_summary', 'question_generation']                     

           INFO     Successfully configured 4 LLM blocks with: model:                           ]8;id=502606;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=868047;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    'hosted_vllm/glm-4.7-fp8', api_base: 'http://10.241.128.25:8100/v1',                           
                    timeout: '600.0'                                                                               

           INFO     Configured blocks: ['answer_generation', 'eval_faithful_llm_chat',          ]8;id=231263;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=161338;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#303\303]8;;\
                    'gen_extractive_summary', 'question_generation']                                               

In [17]:
import dotenv
dotenv.load_dotenv(override=True)

True

In [29]:
import litellm

litellm.completion(
    model="groq/qwen/qwen3-32b",
    api_key=os.environ['GROQ_API_KEY'],
    api_base="https://api.groq.com/openai/v1",
    timeout=600.0,
    messages=[
        {"role": "user", "content": "Hello, how are you?"}
    ]
)


ModelResponse(id='chatcmpl-ad5c68ae-9bfc-47a4-9619-1ca40fe4cf6c', created=1774105516, model='qwen/qwen3-32b', object='chat.completion', system_fingerprint='fp_2bfcc54d36', choices=[Choices(finish_reason='stop', index=0, message=Message(content='<think>\nOkay, the user just says "Hello, how are you?" and I need to respond. First, I should acknowledge their greeting. I can\'t really feel emotions, so I should be honest. Let them know I\'m here and eager to help. Maybe ask how they\'re doing in return. Keep it friendly and open-ended. Add an emoji to keep it light. Let me put it all together.\n</think>\n\nHello! 🌟 I\'m doing well, thank you for asking! How can I assist you today? 😊', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None))], usage=Usage(completion_tokens=111, prompt_tokens=14, total_tokens=125, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.232318545, prompt_time=0.000402223, completion_time=0.246492265, total_time=0

In [31]:
!curl http://10.241.128.25:8100/v1/models

{"object":"list","data":[{"id":"glm-4.7-fp8","object":"model","created":1774116557,"owned_by":"vllm","root":"zai-org/GLM-4.7-FP8","parent":null,"max_model_len":202752,"permission":[{"id":"modelperm-b7887bc99cbd104c","object":"model_permission","created":1774116557,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [35]:

number_of_summaries = int(os.getenv("NUMBER_OF_SUMMARIES", "50"))
# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "question_generation_prompt":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks
# Generate data for extractive summary
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 32768},
        "gen_extractive_summary": {"n": number_of_summaries, "max_tokens": 32768},
    }
else:
    runtime_params = {"gen_extractive_summary": {"n": number_of_summaries}}

extractive_summary_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "extractive_summary"), exist_ok=True)

extractive_summary_generated_data.to_json(
    os.path.join(save_data_path, "extractive_summary" if not generate_cpt else "extractive_summary_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Extractive summary: {len(extractive_summary_generated_data)} records")

print(f"✓ Columns: {list(extractive_summary_generated_data.columns.tolist())}")

[18:10:46] INFO     Using max_concurrency=8 for LLM requests                                       ]8;id=461444;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=111229;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#419\419]8;;\

           INFO     Starting flow 'Extractive Summary Knowledge Tuning Dataset Generation Flow'    ]8;id=438016;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=294385;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#448\448]8;;\
                    v2.0.0 with 10 samples across 20 blocks (max_concurrency=8)                                    

           INFO     Executing block 1/20: duplicate_document_col (DuplicateColumnsBlock)           ]8;id=46284;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=80015;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#220\220]8;;\

╭──────────────────────────────────────────── duplicate_document_col ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: DuplicateColumnsBlock                                                                               │
│ Input Rows: 10                                                                                                  │
│ Input Columns: 8                                                                                                │
│ Column Names: document, source_file, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3,     │
│ domain                                                                                                          │
│ Expected Output Columns: base_document                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── duplicate_document_col - Complete ───────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 10 → 10                                                                                                   │
│ Columns: 8 → 9                                                                                                  │
│ 🟢 Added: base_document                                                                                         │
│ 📋 Final Columns: base_document, document, document_outline, domain, icl_document, icl_query_1, icl_query_2,    │
│ icl_query_3, source_file                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

[19:20:47] ERROR    Failed to generate async responses: litellm.Timeout: Timeout Error:       ]8;id=488527;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=439244;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py#518\518]8;;\
                    Hosted_vllmException - litellm.Timeout: Connection timed out. Timeout                          
                    passed=600.0, time taken=600.105 seconds LiteLLM Retried: 6 times                              

[19:20:47] ERROR    Block 'gen_extractive_summary' failed during execution: litellm.Timeout:       ]8;id=133302;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=640567;file:///workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#292\292]8;;\
                    Timeout Error: Hosted_vllmException - litellm.Timeout: Connection timed out.                   
                    Timeout passed=600.0, time taken=600.105 seconds LiteLLM Retried: 6 times                      

╭───────────────────── Extractive Summary Knowledge Tuning Dataset Generation Flow - Failed ──────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.00s │   10 → 10    │       +1        │     ✓      │           │
│ │ extractive_summary_… │ PromptBuilderB… │      0.01s │   10 → 10    │       +1        │     ✓      │           │
│ │ gen_extractive_summ… │ LLMChatBlock    │   4200.83s │   10 → ❌    │        —        │     ✗      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 3 blocks        │   4200.84s │   0 final    │     0 final     │    2/3     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FlowValidationError: Block 'gen_extractive_summary' execution failed: litellm.Timeout: Timeout Error: Hosted_vllmException - litellm.Timeout: Connection timed out. Timeout passed=600.0, time taken=600.105 seconds LiteLLM Retried: 6 times

In [39]:
print(len(extractive_summary_generated_data))

NameError: name 'extractive_summary_generated_data' is not defined

In [ ]:
# Generate similar data for Detailed Summary
flow_name = flow_names["Detailed Summary Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)

# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "question_generation_prompt":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks

if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 1024},
        "gen_detailed_summary": {"n": number_of_summaries, "max_tokens": 6000},
    }
else:
    runtime_params = {"gen_detailed_summary": {"n": number_of_summaries}}
# Generate data for detailed summary
detailed_summary_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "detailed_summary"), exist_ok=True)

detailed_summary_generated_data.to_json(
    os.path.join(save_data_path, "detailed_summary" if not generate_cpt else "detailed_summary_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Detailed summary: {len(detailed_summary_generated_data)} records")

print(f"✓ Columns: {list(detailed_summary_generated_data.columns.tolist())}")

In [ ]:
# Generate similar data for key facts
flow_name = flow_names["Key Facts Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)


# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "parse_atomic_facts_to_individual_facts":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks

runtime_params = {}
if enable_reasoning:
    # Increase max tokens for Question Generation to accommodate reasoning content
    runtime_params = {
        "generate_key_fact_qa": {"max_tokens": 6000},
    }

# Generate data for key facts summary
key_facts_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "key_facts_to_qa"), exist_ok=True)

key_facts_generated_data.to_json(
    os.path.join(save_data_path, "key_facts_to_qa" if not generate_cpt else "key_facts_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Key facts: {len(key_facts_generated_data)} records")

print(f"✓ Columns: {list(key_facts_generated_data.columns.tolist())}")

In [ ]:
flow_name = flow_names["Document Based Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)
runtime_params = {}
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 2048},
    }

if generate_cpt:
    # Skip generation if CPT is enabled. We simply train on the original document
    document_based_generated_data = quality_corpus[["document", "document_outline", "domain"]]
else:
    document_based_generated_data = flow.generate(
        quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
    )

os.makedirs(os.path.join(save_data_path, "document_based_qa"), exist_ok=True)

document_based_generated_data.to_json(
    os.path.join(save_data_path, "document_based_qa" if not generate_cpt else "document_based_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Document based: {len(document_based_generated_data)} records")

print(f"✓ Columns: {list(document_based_generated_data.columns.tolist())}")

🎉 You now have all three four of document augmentations (detailed summaries, extractive summaries, key facts and document based) along with their corresponding QA pairs.

✅ Next steps:
   - Combine and curate these datasets to prepare your final training data.
   - For detailed guidance on post-processing, mixing, and formatting the data for model training (including conversion to messages format), please refer to [knowledge_mixing.ipynb](knowledge_mixing.ipynb).